In [1]:
from a_slm import transformer

import importlib
import math
from pathlib import Path
import urllib.request

import torch
import torch.nn as nn
from transformers import AutoTokenizer

importlib.reload(transformer)

torch.manual_seed(42)


/Users/desktop/Documents/a-slm/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device:", device)


device: mps


In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    "HuggingFaceTB/SmolLM2-135M"
)

vocab_size = len(tokenizer)

print("vocab size:", vocab_size)
print("eos token:", tokenizer.eos_token)
print("eos id:", tokenizer.eos_token_id)


vocab size: 49152
eos token: <|endoftext|>
eos id: 0


In [4]:
data_path = Path("input.txt")

if not data_path.exists():
    url = (
        "https://raw.githubusercontent.com/karpathy/"
        "char-rnn/master/data/tinyshakespeare/input.txt"
    )
    print("Downloading Tiny Shakespeare...")
    urllib.request.urlretrieve(url, data_path)

text = data_path.read_text(encoding="utf-8")

print("characters:", len(text))
print()
print(text[:500])


characters: 1115394

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [5]:
token_ids = tokenizer.encode(
    text,
    add_special_tokens=False
)

token_ids.append(tokenizer.eos_token_id)

tokens = torch.tensor(
    token_ids,
    dtype=torch.long
)

print("total tokens:", len(tokens))
print("first 30 token ids:", tokens[:30].tolist())
print(
    "first 30 tokens:",
    tokenizer.convert_ids_to_tokens(tokens[:30].tolist())
)


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (341094 > 8192). Running this sequence through the model will result in indexing errors


total tokens: 341095
first 30 token ids: [5345, 32062, 42, 198, 6121, 392, 7219, 750, 2030, 28, 4875, 549, 3287, 30, 198, 198, 4518, 42, 198, 15024, 494, 28, 3287, 30, 198, 198, 5345, 32062, 42, 198]
first 30 tokens: ['First', 'ĠCitizen', ':', 'Ċ', 'Before', 'Ġwe', 'Ġproceed', 'Ġany', 'Ġfurther', ',', 'Ġhear', 'Ġme', 'Ġspeak', '.', 'Ċ', 'Ċ', 'All', ':', 'Ċ', 'Spe', 'ak', ',', 'Ġspeak', '.', 'Ċ', 'Ċ', 'First', 'ĠCitizen', ':', 'Ċ']


In [6]:
split_idx = int(0.9 * len(tokens))

train_tokens = tokens[:split_idx]
val_tokens = tokens[split_idx:]

print("train tokens:", len(train_tokens))
print("val tokens:  ", len(val_tokens))


train tokens: 306985
val tokens:   34110


In [7]:
batch_size = 4
context_length = 64

def get_batch(data, batch_size, context_length, device):
    max_start = len(data) - context_length - 1

    starts = torch.randint(
        0,
        max_start + 1,
        (batch_size,)
    )

    x = torch.stack([
        data[i:i + context_length]
        for i in starts
    ])

    y = torch.stack([
        data[i + 1:i + context_length + 1]
        for i in starts
    ])

    return x.to(device), y.to(device)


In [8]:
x, y = get_batch(
    train_tokens,
    batch_size=batch_size,
    context_length=context_length,
    device=device
)

print("x shape:", x.shape)
print("y shape:", y.shape)

assert x.shape == (batch_size, context_length)
assert y.shape == (batch_size, context_length)

print()
print("Input sample:")
print(tokenizer.decode(x[0].tolist()))

print()
print("Target sample:")
print(tokenizer.decode(y[0].tolist()))


x shape: torch.Size([4, 64])
y shape: torch.Size([4, 64])

Input sample:
 you well:
Incapable and shallow innocents,
You cannot guess who caused your father's death.

Boy:
Grandam, we can; for my good uncle Gloucester
Told me, the king, provoked by the queen,
Devised impeachments to imprison him :


Target sample:
 well:
Incapable and shallow innocents,
You cannot guess who caused your father's death.

Boy:
Grandam, we can; for my good uncle Gloucester
Told me, the king, provoked by the queen,
Devised impeachments to imprison him :
And


In [9]:
model = transformer.Transformer(
    num_l4g_blocks=6,
    hidden_size=48,
    intermediate_size=128,
    num_q_heads=6,
    num_kv_heads=2,
    vocab_size=vocab_size,
    window_size=3
).to(device)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


Total parameters:     3,099,984
Trainable parameters: 3,099,984


In [10]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

loss_fn = nn.CrossEntropyLoss()


In [11]:
model.eval()

with torch.no_grad():
    x, y = get_batch(
        train_tokens,
        batch_size=batch_size,
        context_length=context_length,
        device=device
    )

    logits = model(x)

    initial_loss = loss_fn(
        logits.reshape(-1, vocab_size),
        y.reshape(-1)
    )

print(f"initial loss:       {initial_loss.item():.4f}")
print(f"ln(vocab_size):     {math.log(vocab_size):.4f}")


initial loss:       31.2769
ln(vocab_size):     10.8027


In [12]:
@torch.no_grad()
def estimate_loss(
    model,
    train_tokens,
    val_tokens,
    loss_fn,
    batch_size,
    context_length,
    device,
    eval_batches=20
):
    model.eval()

    losses = {}

    for split_name, data in [
        ("train", train_tokens),
        ("val", val_tokens)
    ]:
        split_losses = []

        for _ in range(eval_batches):
            x, y = get_batch(
                data,
                batch_size=batch_size,
                context_length=context_length,
                device=device
            )

            logits = model(x)

            loss = loss_fn(
                logits.reshape(-1, vocab_size),
                y.reshape(-1)
            )

            split_losses.append(loss.item())

        losses[split_name] = (
            sum(split_losses) / len(split_losses)
        )

    model.train()

    return losses


In [13]:
num_steps = 500
eval_interval = 50

model.train()

for step in range(num_steps + 1):

    if step % eval_interval == 0:
        losses = estimate_loss(
            model=model,
            train_tokens=train_tokens,
            val_tokens=val_tokens,
            loss_fn=loss_fn,
            batch_size=batch_size,
            context_length=context_length,
            device=device,
            eval_batches=10
        )

        print(
            f"step {step:4d} | "
            f"train {losses['train']:.4f} | "
            f"val {losses['val']:.4f}"
        )

    if step == num_steps:
        break

    x, y = get_batch(
        train_tokens,
        batch_size=batch_size,
        context_length=context_length,
        device=device
    )

    optimizer.zero_grad()

    logits = model(x)

    loss = loss_fn(
        logits.reshape(-1, vocab_size),
        y.reshape(-1)
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()


step    0 | train 31.5757 | val 31.1875
step   50 | train 17.3178 | val 16.8953
step  100 | train 13.6835 | val 13.5410
step  150 | train 10.8020 | val 11.0472
step  200 | train 9.1260 | val 9.2324
step  250 | train 8.0514 | val 8.2882
step  300 | train 7.2332 | val 7.6955
step  350 | train 6.6951 | val 7.0837
step  400 | train 6.4635 | val 6.8168
step  450 | train 6.1410 | val 6.4355
step  500 | train 6.0762 | val 6.5817


In [14]:
@torch.no_grad()
def generate(
    model,
    tokenizer,
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_k=40
):
    model.eval()

    generated = tokenizer.encode(
        prompt,
        add_special_tokens=False,
        return_tensors="pt"
    ).to(device)

    for _ in range(max_new_tokens):

        logits = model(generated)

        next_token_logits = logits[:, -1, :]

        next_token_logits = (
            next_token_logits / temperature
        )

        if top_k is not None:
            k = min(top_k, next_token_logits.shape[-1])

            values, _ = torch.topk(
                next_token_logits,
                k
            )

            cutoff = values[:, [-1]]

            next_token_logits = torch.where(
                next_token_logits < cutoff,
                torch.full_like(
                    next_token_logits,
                    float("-inf")
                ),
                next_token_logits
            )

        probs = torch.softmax(
            next_token_logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        generated = torch.cat(
            [generated, next_token],
            dim=1
        )

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(
        generated[0].tolist(),
        skip_special_tokens=False
    )


In [15]:
sample = generate(
    model=model,
    tokenizer=tokenizer,
    prompt="ROMEO:",
    max_new_tokens=120,
    temperature=0.8,
    top_k=40
)

print(sample)


ROMEO:
 I have you, but a I for an'd,
You have shall for your to
And
CORIOLANUS:
Ibury,
Nay, well my,
And:
And me, be aT,
 Milk, to.

MENENIUS:
 one your a, my have have!

 Syntheticant:
I am to
 with him.

Firstars:
 said is,
When.

MENENIUS:
When think: by this,
 I will of his
Nay
